ML Assignment - 2

In [1]:
# Use requirements.txt file to install any missing packages:
# pip install -r requirements.txt

import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, 
    recall_score, f1_score, matthews_corrcoef
)

import warnings
warnings.filterwarnings('ignore')

# Package to fetch dataset from UCI repo
from ucimlrepo import fetch_ucirepo


In [ ]:
# Fetch and prepare the Dataset
print("Fetching the Obesity dataset from UCI repository...")
obesity = fetch_ucirepo(id=544)
X = obesity.data.features
y_raw = obesity.data.targets['NObeyesdad']

print("Dataset fetched successfully")

# Encode target labels to integers for multi-class classification
le = LabelEncoder()
y = pd.Series(le.fit_transform(y_raw), name='NObeyesdad')

# Save the LabelEncoder to use later
os.makedirs('model', exist_ok=True)
joblib.dump(le, 'model/label_encoder.joblib')

# Handle missing values, if any (converting '?' to NaN in dataset)
X = X.replace('?', np.nan)

# Perform 80-20 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Save test data to test_data.csv
test_data = X_test.copy()
# Convert labels back to original encoding
test_data['NObeyesdad'] = le.inverse_transform(y_test)
test_data.to_csv('test_data.csv', index=False)
print("Test data saved to test_data.csv")

# Preprocessing pipeline
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Define the 5 Models
# SVC probability=True is required to calculate AUC Score
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(), # Gaussian used as StandardScaler produces negative values
    'Support Vector Machine': SVC(probability=True, random_state=42) 
}

# Ensure model directory exists
os.makedirs('model', exist_ok=True)

# Train, Evaluate, and Save the 5 Models
results = []

for name, clf in models.items():
    print(f"\nTraining dataset with {name} model...")
    
    # Create a full pipeline containing preprocessing and the classifier
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('classifier', clf)])
    
    # Train
    pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test) 
    
    # Calculate Metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='weighted')
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    mcc = matthews_corrcoef(y_test, y_pred)
    
    results.append({
        'ML Model Name': name,
        'Accuracy': round(acc, 4),
        'AUC': round(auc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'MCC': round(mcc, 4)
    })
    
    # Save the trained pipeline
    model_filename = f"model/{name.replace(' ', '_').lower()}.joblib"
    joblib.dump(pipeline, model_filename)
    print(f"Saved {name} to {model_filename}")

# Display Final Metrics Table
print("\n--- Evaluation Metrics ---")
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


Fetching the Obesity dataset from UCI repository...
Dataset fetched successfully
Test data saved to test_data.csv

Training dataset with Logistic Regression model...
Saved Logistic Regression to model/logistic_regression.joblib

Training dataset with Decision Tree model...
Saved Decision Tree to model/decision_tree.joblib

Training dataset with KNN model...
Saved KNN to model/knn.joblib

Training dataset with Naive Bayes model...
Saved Naive Bayes to model/naive_bayes.joblib

Training dataset with Support Vector Machine model...
Saved Support Vector Machine to model/support_vector_machine.joblib

--- Evaluation Metrics ---
         ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.8747 0.9837     0.8729  0.8747 0.8731 0.8539
         Decision Tree    0.9149 0.9504     0.9166  0.9149 0.9153 0.9007
                   KNN    0.8251 0.9661     0.8248  0.8251 0.8170 0.7981
           Naive Bayes    0.5083 0.8382     0.5275  0.5083 0.4607 0.4471
Supp